# KIRA — Training Model Penyakit Daun Cabai (Kaggle)

Notebook ini **hanya** memakai dataset cabai `dhenyd/chili-plant-disease`
(5 kelas: `healthy`, `leaf curl`, `leaf spot`, `whitefly`, `yellowish`).
Tidak ada data tomat.

Output: `kira_ml_model.joblib` yang formatnya cocok dengan `kira-ml-service/main.py`
(key `model`, `xgb`, `le`, `classes`).

## Cara pakai di Kaggle
1. **Create → Notebook**.
2. Panel kanan **Add Input** → cari **`chili-plant-disease`** (oleh `dhenyd`) → **Add**.
3. Panel kanan **Session options / Accelerator** → pilih **GPU T4 x2** (atau P100).
4. **Run All**. Estimasi ±5–15 menit.
5. Setelah selesai, buka **Output** di panel kanan → download
   `kira_ml_model.joblib`, lalu ganti file di `kira-ml-service/kira_ml_model.joblib`.

Preprocessing di notebook ini **harus sama** dengan service:
`Resize(192,192) → ToTensor → Normalize(ImageNet)`.

In [ ]:
!pip -q install "xgboost>=2" scikit-learn timm torch torchvision matplotlib seaborn pillow joblib

In [ ]:
import glob
import os
import random
from collections import Counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import timm
import torch
import xgboost as xgb
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torchvision import transforms

SEED = 42
IMG_SIZE = 192
MODEL_NAME = "mobilenetv3_large_100"
AUG_PER_IMAGE = int(os.environ.get("KIRA_AUG_PER_IMAGE", "4"))  # 0 = tanpa augmentasi
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)

## 1. Temukan dataset cabai

Kalau dataset belum muncul, pastikan sudah **Add Input** di panel kanan.
Cell ini mencari otomatis folder yang punya subfolder `train/`, dan hanya
menerima folder yang isinya bukan tomat.

In [ ]:
def looks_like_dataset(path):
    if not os.path.isdir(path):
        return False
    entries = {e.lower() for e in os.listdir(path)}
    return "train" in entries


candidates = []
for base in ("/kaggle/input", "/kaggle/input/datasets"):
    if not os.path.isdir(base):
        continue
    for root in glob.glob(base + "/*") + glob.glob(base + "/*/*"):
        if looks_like_dataset(root):
            candidates.append(root)

# fallback: dataset di-download via kagglehub
if not candidates:
    try:
        import kagglehub
        p = kagglehub.dataset_download("dhenyd/chili-plant-disease")
        candidates = [p] if looks_like_dataset(p) else []
    except Exception as exc:
        print("kagglehub gagal:", exc)

env_dir = os.environ.get("KIRA_DATA_DIR")
if env_dir and looks_like_dataset(env_dir):
    candidates.insert(0, env_dir)

candidates = sorted(set(candidates), key=lambda p: 0 if p == env_dir else 1)
print("kandidat dataset:")
for c in candidates:
    print("  -", c)

if not candidates:
    raise FileNotFoundError(
        "Dataset cabai tidak ditemukan. Klik Add Input lalu cari 'chili-plant-disease' (dhenyd)."
    )

# Pilih kandidat pertama yang punya kelas cabai, bukan tomat.
DATA_DIR = None
for c in candidates:
    names = sorted(os.listdir(os.path.join(c, "train")))
    if any("tomato" in n.lower() for n in names):
        print("lewati (ada tomat):", c)
        continue
    DATA_DIR = c
    break

if DATA_DIR is None:
    raise ValueError("Semua kandidat berisi data tomat. Tambahkan dataset cabai yang benar.")

print("\nDATA_DIR:", DATA_DIR)
print("split:", sorted(os.listdir(DATA_DIR)))
print("kelas train:", sorted(os.listdir(os.path.join(DATA_DIR, "train"))))

In [ ]:
SPLIT_ALIASES = {
    "train": ["train"],
    "val": ["val", "valid", "validation"],
    "test": ["test"],
}


def find_split_dir(data_dir, split):
    for name in SPLIT_ALIASES[split]:
        path = os.path.join(data_dir, name)
        if os.path.isdir(path):
            return path
    return None


def collect_split(data_dir, split):
    root = find_split_dir(data_dir, split)
    if root is None:
        return [], []

    paths, labels = [], []
    for class_name in sorted(os.listdir(root)):
        class_dir = os.path.join(root, class_name)
        if not os.path.isdir(class_dir):
            continue
        if class_name.lower() in {"train", "val", "valid", "validation", "test"}:
            continue

        images = []
        for ext in IMAGE_EXTS:
            images.extend(glob.glob(os.path.join(class_dir, "**", f"*{ext}"), recursive=True))
            images.extend(glob.glob(os.path.join(class_dir, "**", f"*{ext.upper()}"), recursive=True))
        images = sorted(set(images))

        label = f"chili_{class_name.strip().replace(' ', '_').lower()}"
        for path in images:
            paths.append(path)
            labels.append(label)

    return paths, labels


train_paths, train_labels = collect_split(DATA_DIR, "train")
val_paths, val_labels = collect_split(DATA_DIR, "val")
test_paths, test_labels = collect_split(DATA_DIR, "test")

# Kalau dataset tidak punya val/test, pecah dari train 80/10/10.
if not val_paths or not test_paths:
    from sklearn.model_selection import train_test_split

    all_paths = np.array(train_paths)
    all_labels = np.array(train_labels)
    idx = np.arange(len(all_paths))
    idx_tmp, idx_test = train_test_split(idx, test_size=0.1, random_state=SEED, stratify=all_labels)
    idx_train, idx_val = train_test_split(idx_tmp, test_size=0.1111, random_state=SEED, stratify=all_labels[idx_tmp])
    train_paths = all_paths[idx_train].tolist()
    train_labels = all_labels[idx_train].tolist()
    val_paths = all_paths[idx_val].tolist()
    val_labels = all_labels[idx_val].tolist()
    test_paths = all_paths[idx_test].tolist()
    test_labels = all_labels[idx_test].tolist()
    print("Dataset tanpa val/test -> split manual 80/10/10")

assert train_paths, "Tidak ada gambar train."
assert all("tomato" not in lab.lower() for lab in train_labels + val_labels + test_labels), "Data tomat tidak boleh dipakai."

print("train:", len(train_paths))
print("val  :", len(val_paths))
print("test :", len(test_paths))
print("kelas:", sorted(set(train_labels)))

In [ ]:
all_counts = Counter(train_labels + val_labels + test_labels)
plt.figure(figsize=(8, 4))
plt.bar(all_counts.keys(), all_counts.values(), color="#d4a373")
plt.xticks(rotation=25, ha="right")
plt.title("Distribusi Kelas Cabai")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, path, label in zip(axes.flatten(), train_paths[:10], train_labels[:10]):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(label.replace("chili_", ""), fontsize=9)
    ax.axis("off")
plt.suptitle("Contoh Gambar Daun Cabai")
plt.tight_layout()
plt.show()

## 2. Ekstraksi fitur MobileNetV3

Semua fitur memakai transform yang sama dengan service. Label ikut disimpan
per gambar supaya gambar rusak yang di-skip tidak membuat label bergeser.

In [ ]:
base_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

aug_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.15),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

extractor = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0).to(device).eval()
print("extractor siap:", MODEL_NAME)

In [ ]:
def embed_labeled(paths, labels, tf, batch_size=32):
    """Return (features, labels) yang selalu sejajar walau ada file rusak."""
    feats, kept_labels = [], []
    batch, batch_labels = [], []

    def flush():
        if not batch:
            return
        tensor = torch.stack(batch).to(device)
        with torch.no_grad():
            out = extractor(tensor).cpu().numpy()
        feats.append(out)
        kept_labels.extend(batch_labels)
        batch.clear()
        batch_labels.clear()

    for path, label in zip(paths, labels):
        try:
            image = Image.open(path).convert("RGB")
            tensor = tf(image)
        except Exception as exc:
            print("[skip]", os.path.basename(path), exc)
            continue
        batch.append(tensor)
        batch_labels.append(label)
        if len(batch) >= batch_size:
            flush()
    flush()

    if not feats:
        return np.empty((0, 0), dtype=np.float32), []
    return np.concatenate(feats, axis=0), kept_labels


print("Ekstraksi train (asli)...")
x_train, y_train_kept = embed_labeled(train_paths, train_labels, base_tf)

if AUG_PER_IMAGE > 0:
    print(f"Ekstraksi train ({AUG_PER_IMAGE}x augmentasi)...")
    x_aug, y_aug = embed_labeled(train_paths * AUG_PER_IMAGE, train_labels * AUG_PER_IMAGE, aug_tf)
else:
    x_aug, y_aug = np.empty((0, 0), dtype=np.float32), []

print("Ekstraksi val...")
x_val, y_val_kept = embed_labeled(val_paths, val_labels, base_tf)

print("Ekstraksi test...")
x_test, y_test_kept = embed_labeled(test_paths, test_labels, base_tf)

print("shape:", x_train.shape, x_aug.shape, x_val.shape, x_test.shape)

## 3. Training model

Dua kandidat: XGBoost dan LogisticRegression. Model dipilih dari akurasi
validation set, lalu dievaluasi sekali di test set.

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_kept)
y_aug_enc = le.transform(y_aug) if len(y_aug) else np.array([], dtype=int)
y_val_enc = le.transform(y_val_kept)
y_test_enc = le.transform(y_test_kept)

x_train_fit = np.concatenate([x_train, x_aug], axis=0) if len(x_aug) else x_train
y_train_fit = np.concatenate([y_train_enc, y_aug_enc]) if len(y_aug_enc) else y_train_enc

print("kelas:", list(le.classes_))
print("shape train fit:", x_train_fit.shape)

In [ ]:
xgb_params = dict(
    n_estimators=700,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=SEED,
)

try:
    xgb_model = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=60)
    xgb_model.fit(x_train_fit, y_train_fit, eval_set=[(x_val, y_val_enc)], verbose=False)
except TypeError:
    # xgboost 2.x memindahkan early_stopping_rounds ke constructor berbeda
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(x_train_fit, y_train_fit, verbose=False)

logreg_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000, C=1.0, n_jobs=-1, random_state=SEED),
)
logreg_model.fit(x_train_fit, y_train_fit)

models = {"xgboost": xgb_model, "logreg": logreg_model}
val_scores = {}
for name, model in models.items():
    pred = model.predict(x_val)
    val_scores[name] = accuracy_score(y_val_enc, pred)
    print(f"{name} val accuracy: {val_scores[name]:.4f}")

best_name = max(val_scores, key=val_scores.get)
best_model = models[best_name]
print("model terpilih:", best_name)

In [ ]:
pred_test = best_model.predict(x_test)
print("test accuracy:", accuracy_score(y_test_enc, pred_test))
print(classification_report(y_test_enc, pred_test, target_names=le.classes_, digits=3, zero_division=0))

cm = confusion_matrix(y_test_enc, pred_test)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.title(f"Confusion Matrix - {best_name}")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Simpan model

Format bundle harus sama dengan yang dibaca `kira-ml-service/main.py`.

In [ ]:
out_path = "/kaggle/working/kira_ml_model.joblib"
bundle = {
    "model": best_model,
    "xgb": xgb_model,
    "le": le,
    "classes": list(le.classes_),
}
joblib.dump(bundle, out_path, compress=3)
print("disimpan:", out_path)
print("ukuran: %.1f KB" % (os.path.getsize(out_path) / 1024))
print("classes:", bundle["classes"])

In [ ]:
# Sanity check: load ulang lalu prediksi 1 gambar test
loaded = joblib.load(out_path)
loaded_model = loaded["model"]
loaded_le = loaded["le"]

sample_path = test_paths[0]
feat, _ = embed_labeled([sample_path], [test_labels[0]], base_tf)
pred = int(np.argmax(loaded_model.predict_proba(feat)[0]))
print("file     :", sample_path)
print("aktual   :", test_labels[0])
print("prediksi :", loaded_le.classes_[pred])

## 5. Unduh model

1. Buka tab **Output** di panel kanan notebook.
2. Download **`kira_ml_model.joblib`**.
3. Timpa `kira-ml-service/kira_ml_model.joblib` di repo.
4. Redeploy service ML (Railway) supaya model baru dipakai.

Cek cepat setelah deploy:

```bash
curl https://<service-ml>/health
```

Harus muncul 5 class: `chili_healthy`, `chili_leaf_curl`, `chili_leaf_spot`,
`chili_whitefly`, `chili_yellowish`.